# Node2Vec Baseline Demo

This notebook demonstrates the node2vec link-prediction baseline on top of a saved split artifact.

It shows how to:

- load a deterministic split artifact
- define a CPU-friendly node2vec configuration
- train train-graph-only node embeddings
- evaluate the link-prediction baseline with the shared result schema


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

In [ ]:
from citegnn.baselines.node2vec import Node2VecConfig, run_node2vec_baseline
from citegnn.data.splits import load_split_artifact


In [ ]:
dataset_name = "cora"
seed = 42
split_path = PROJECT_ROOT / "artifacts" / "splits" / f"{dataset_name}_seed{seed}_lp.pt"

if not split_path.exists():
    raise FileNotFoundError(
        f"Missing split artifact at {split_path}. Run scripts/build_link_prediction_split.py first."
    )

split = load_split_artifact(split_path)

{
    "dataset": split.dataset_name,
    "seed": split.seed,
    "train_graph_edges": int(split.train_graph_edge_index.size(1)),
    "train_supervision_edges": int(split.train_pos_edge_index.size(1) + split.train_neg_edge_index.size(1)),
    "val_examples": int(split.val_pos_edge_index.size(1) + split.val_neg_edge_index.size(1)),
    "test_examples": int(split.test_pos_edge_index.size(1) + split.test_neg_edge_index.size(1)),
}

In [ ]:
config = Node2VecConfig(
    embedding_dim=16,
    walk_length=10,
    walks_per_node=2,
    context_window=5,
    negative_samples=3,
    epochs=1,
    batch_size=256,
)

config.to_dict()

In [ ]:
result = run_node2vec_baseline(split, config=config)
result.to_dict()

To switch datasets, change `dataset_name` to `"pubmed"` and rerun the notebook.

For the CLI entrypoint, use `python scripts/run_node2vec_baseline.py --split-path artifacts/splits/cora_seed42_lp.pt`.